In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

# 資產設定
TICKERS = ["AAPL", "SPY", "TLT", "GLD", "VNQ"]

# 一年交易日假設
TRADING_DAYS = 252

pd.set_option("display.float_format", lambda x: f"{x:.4f}")

print("分析資產：", TICKERS)

In [ ]:
# 明確設定 auto_adjust=False，確保可以取得 Adjusted Close
raw_data = yf.download(
    tickers=TICKERS,
    period="5y",
    interval="1d",
    auto_adjust=False,
    progress=False,
    group_by="column",
    threads=True
)

if raw_data.empty:
    raise RuntimeError("無法下載資料，請確認網路連線或稍後再試。")

# 兼容 yfinance 不同版本回傳的 MultiIndex 格式
if isinstance(raw_data.columns, pd.MultiIndex):
    level_0 = raw_data.columns.get_level_values(0)
    level_1 = raw_data.columns.get_level_values(1)

    if "Adj Close" in level_0:
        prices = raw_data["Adj Close"].copy()
    elif "Adj Close" in level_1:
        prices = raw_data.xs("Adj Close", axis=1, level=1).copy()
    else:
        raise KeyError("下載資料中找不到 Adjusted Close 欄位。")
else:
    if "Adj Close" not in raw_data.columns:
        raise KeyError("下載資料中找不到 Adjusted Close 欄位。")
    prices = raw_data[["Adj Close"]].copy()
    prices.columns = [TICKERS[0]]

# 確認資產欄位完整
missing_tickers = [ticker for ticker in TICKERS if ticker not in prices.columns]

if missing_tickers:
    raise ValueError(f"以下資產沒有成功下載：{missing_tickers}")

prices = prices[TICKERS]

# 整理日期索引
prices.index = pd.to_datetime(prices.index)

if prices.index.tz is not None:
    prices.index = prices.index.tz_localize(None)

prices = prices.sort_index()
prices = prices.loc[~prices.index.duplicated(keep="last")]

# 將非數值資料轉換成 NaN，再刪除缺失值
prices = prices.apply(pd.to_numeric, errors="coerce")
prices = prices.dropna(how="any")

print(f"資料期間：{prices.index.min().date()} 至 {prices.index.max().date()}")
print(f"有效交易日數：{len(prices)}")

display(prices.head())

In [ ]:
# 計算每日簡單報酬率
return_matrix = prices.pct_change(fill_method=None).dropna(how="any")

print("Return Matrix：")
display(return_matrix.head())

print("\n缺失值檢查：")
display(return_matrix.isna().sum())

In [ ]:
# 每日平均報酬率
average_daily_return = return_matrix.mean()

# 年化報酬率：使用幾何複利計算
annualized_return = (
    (1 + return_matrix).prod()
    ** (TRADING_DAYS / len(return_matrix))
    - 1
)

# 每日標準差
daily_std = return_matrix.std()

# 年化波動度
annualized_volatility = daily_std * np.sqrt(TRADING_DAYS)

# 建立分析表
performance_summary = pd.DataFrame({
    "Average Daily Return": average_daily_return,
    "Annualized Return": annualized_return,
    "Daily Standard Deviation": daily_std,
    "Annualized Volatility": annualized_volatility
})

print("各資產報酬與風險分析：")
display(performance_summary.style.format("{:.2%}"))

In [ ]:
# 每日共變異數矩陣
daily_covariance_matrix = return_matrix.cov()

# 年化共變異數矩陣
annualized_covariance_matrix = daily_covariance_matrix * TRADING_DAYS

# 相關係數矩陣
correlation_matrix = return_matrix.corr()

print("Daily Covariance Matrix：")
display(daily_covariance_matrix.round(6))

print("Annualized Covariance Matrix：")
display(annualized_covariance_matrix.round(6))

print("Correlation Matrix：")
display(correlation_matrix.round(4))

In [ ]:
plt.figure(figsize=(9, 7))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Correlation"}
)

plt.title("Asset Return Correlation Matrix", fontsize=14)
plt.xlabel("Asset")
plt.ylabel("Asset")
plt.tight_layout()
plt.show()

In [ ]:
# 取出相關係數矩陣上三角，避免重複計算
upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

pairwise_correlations = upper_triangle.stack().sort_values()

def format_pairs(pair_series):
    result = []
    for (asset_1, asset_2), value in pair_series.items():
        result.append(f"{asset_1} 與 {asset_2}（相關係數 {value:.3f}）")
    return "、".join(result)

# Q1：最高相關性
highest_pair = pairwise_correlations.tail(1)
highest_pair_text = format_pairs(highest_pair)

# Q2：最低或負相關性
negative_pairs = pairwise_correlations[pairwise_correlations < 0]

if len(negative_pairs) > 0:
    lowest_pair_text = format_pairs(negative_pairs.head(3))
    q2_text = (
        f"相關性較低或呈負相關的資產包括：{lowest_pair_text}。"
        "這些資產的價格變動方向較不一致，可能有助於降低投資組合的整體波動。"
    )
else:
    lowest_pairs = pairwise_correlations.head(3)
    lowest_pair_text = format_pairs(lowest_pairs)
    q2_text = (
        f"本次資料中沒有明顯的負相關配對；相關性最低的配對包括："
        f"{lowest_pair_text}。"
        "雖然不是負相關，但較低的相關係數仍可能提供一定的分散效果。"
    )

# 計算各資產與其他資產的平均相關係數
average_correlation = (
    correlation_matrix
    .where(~np.eye(len(correlation_matrix), dtype=bool))
    .mean(axis=1)
    .sort_values()
)

diversification_assets = average_correlation.head(2)

q3_text = (
    "從平均相關係數來看，"
    + "、".join(
        [
            f"{asset}（平均相關係數 {value:.3f}）"
            for asset, value in diversification_assets.items()
        ]
    )
    + "與其他資產的相關性相對較低，因此可能具有較好的 diversification benefit。"
    "不過，實際投資組合仍應同時考慮報酬率、波動度、共變異數與投資權重。"
)

display(Markdown(f"""
## 自動產生的申論題參考草稿

### Q1. 哪些資產彼此相關性最高？

本次資料中，相關性最高的資產配對為：**{highest_pair_text}**。
這表示兩項資產的報酬率變動方向較為一致，將兩者同時納入投資組合時，分散風險的效果可能相對有限。

### Q2. 哪些資產具有較低或負相關性？

{q2_text}

### Q3. 從 Correlation Matrix 判斷，哪些資產可能具有較好的 diversification benefit？

{q3_text}
"""))

In [ ]:
# ==========================================
# Part 3：Monte Carlo Portfolio Simulation
# ==========================================

# 模擬組合數量
N_PORTFOLIOS = 10_000

# 確保資產順序一致
expected_returns = (
    performance_summary
    .loc[TICKERS, "Annualized Return"]
    .to_numpy()
)

annualized_cov_matrix = (
    annualized_covariance_matrix
    .loc[TICKERS, TICKERS]
    .to_numpy()
)

# 固定亂數種子，讓每次執行結果可重現
rng = np.random.default_rng(42)

# 使用 Dirichlet 分布產生權重
# 每一列權重總和必定等於 1，且不允許放空
weights = rng.dirichlet(
    alpha=np.ones(len(TICKERS)),
    size=N_PORTFOLIOS
)

# 檢查權重總和
print("前 5 組權重總和：")
print(weights[:5].sum(axis=1))

# 計算每組投資組合的年化報酬率
portfolio_returns = weights @ expected_returns

# 計算每組投資組合的年化變異數
portfolio_variances = np.einsum(
    "ij,jk,ik->i",
    weights,
    annualized_cov_matrix,
    weights
)

# 避免極小的浮點數誤差造成負數
portfolio_variances = np.maximum(portfolio_variances, 0)

# 計算每組投資組合的年化波動率
portfolio_volatilities = np.sqrt(portfolio_variances)

# 建立模擬結果 DataFrame
portfolio_results = pd.DataFrame({
    "Expected Annual Return": portfolio_returns,
    "Annualized Volatility": portfolio_volatilities
})

# 加入每組投資組合的資產權重
weights_df = pd.DataFrame(weights, columns=TICKERS)
portfolio_results = pd.concat(
    [weights_df, portfolio_results],
    axis=1
)

print(f"\n已完成 {N_PORTFOLIOS:,} 組投資組合模擬。")
display(portfolio_results.head())

In [ ]:
# ==========================================
# Part 4：Minimum Variance Portfolio
# ==========================================

# 找出波動率最低的投資組合
mvp_index = portfolio_results["Annualized Volatility"].idxmin()
mvp = portfolio_results.loc[mvp_index]

# 取得 MVP 的資產權重
mvp_weights = mvp[TICKERS]

# MVP 的報酬率與波動率
mvp_return = mvp["Expected Annual Return"]
mvp_volatility = mvp["Annualized Volatility"]

print("Minimum Variance Portfolio（MVP）結果")
print("-" * 45)
print(f"預期年化報酬率：{mvp_return:.2%}")
print(f"預期年化波動率：{mvp_volatility:.2%}")

# 整理 MVP 資產配置建議
mvp_allocation = pd.DataFrame({
    "Asset": TICKERS,
    "MVP Weight": mvp_weights.to_numpy()
})

print("\nMVP 資產配置建議：")
display(
    mvp_allocation.style.format({
        "MVP Weight": "{:.2%}"
    })
)

print(f"權重總和：{mvp_weights.sum():.2%}")

In [ ]:
# 繪製投資組合機會集合
plt.figure(figsize=(11, 7))

scatter = plt.scatter(
    portfolio_results["Annualized Volatility"],
    portfolio_results["Expected Annual Return"],
    c=portfolio_results["Expected Annual Return"],
    cmap="viridis",
    alpha=0.55,
    s=15,
    label="Simulated Portfolios"
)

# 標示 MVP
plt.scatter(
    mvp_volatility,
    mvp_return,
    color="red",
    marker="*",
    s=300,
    edgecolors="black",
    linewidths=1,
    label="Minimum Variance Portfolio (MVP)",
    zorder=5
)

# 在圖上標示 MVP 數值
plt.annotate(
    f"MVP\nReturn = {mvp_return:.2%}\nVolatility = {mvp_volatility:.2%}",
    xy=(mvp_volatility, mvp_return),
    xytext=(15, 15),
    textcoords="offset points",
    fontsize=10,
    color="red",
    bbox=dict(
        boxstyle="round,pad=0.4",
        facecolor="white",
        edgecolor="red",
        alpha=0.85
    )
)

plt.colorbar(scatter, label="Expected Annual Return")
plt.xlabel("Annualized Volatility")
plt.ylabel("Expected Annual Return")
plt.title("Portfolio Opportunity Set and Minimum Variance Portfolio")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# Part 5：依報告架構顯示圖表與自動產出報告
# ==========================================

from IPython.display import Markdown, display

# ---------- 一、顯示研究目的與資料設定 ----------

display(Markdown(f"""
# 現代投資組合理論（MPT）與理財機器人實作分析報告

## 一、實作目的與投資組合設定

本次實作使用現代投資組合理論（Modern Portfolio Theory, MPT），
透過 Python 與 `yfinance` 下載市場資料，分析不同資產的報酬率、
波動度與相關性，並使用蒙地卡羅模擬法建立投資組合機會集合。

本次選用的資產包括：**{", ".join(TICKERS)}**。

資料期間為 **{prices.index.min().date()} 至 {prices.index.max().date()}**，
共使用 **{len(return_matrix):,} 個交易日**資料。

本次分析流程包括：

1. 下載調整後收盤價。
2. 計算每日報酬率。
3. 計算年化報酬率與年化波動率。
4. 建立相關係數矩陣與共變異數矩陣。
5. 模擬 {len(portfolio_results):,} 組投資組合。
6. 找出最低變異投資組合（MVP）。

---

## 二、資產相關性與風險分析

### 各資產年化報酬率與波動率

| 資產 | 年化報酬率 | 年化波動率 |
|---|---:|---:|
{performance_table}

### Correlation Matrix Heatmap
"""))

# ---------- 二、Heatmap：放在相關性分析段落內 ----------

fig_heatmap, ax_heatmap = plt.subplots(figsize=(8, 6))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax_heatmap,
    cbar_kws={"label": "Correlation"}
)

ax_heatmap.set_title("Correlation Matrix Heatmap")
ax_heatmap.set_xlabel("Asset")
ax_heatmap.set_ylabel("Asset")

fig_heatmap.tight_layout()
plt.show()

print("✅ Heatmap 已成功產生，並放置於資產相關性分析段落。")


# ---------- 三、利用 Heatmap 結果回答 Q1–Q3 ----------

if lowest_corr < 0:
    lowest_description = "呈現負相關"
else:
    lowest_description = "為所有資產配對中最低的相關性"

display(Markdown(f"""
### Q1：哪些資產彼此相關性最高？

本次分析中，相關性最高的資產組合為：

**{highest_asset_1} 與 {highest_asset_2}，相關係數為 {highest_corr:.3f}。**

這表示兩項資產的報酬率變動方向較為一致，
因此同時持有這兩項資產時，分散風險的效果可能相對有限。

### Q2：哪些資產具有較低或負相關性？

本次分析中，相關性最低的資產組合為：

**{lowest_asset_1} 與 {lowest_asset_2}，相關係數為 {lowest_corr:.3f}。**

此結果表示兩項資產的報酬率變動方向較不一致，
且該組合{lowest_description}。
因此將兩者搭配於投資組合中，可能有助於降低整體波動率。

### Q3：哪些資產可能具有較好的 diversification benefit？

根據各資產與其他資產的平均相關係數，
較可能提供分散效果的資產為：

**{diversification_text}。**

這些資產與其他資產的平均相關性較低，
因此可能在投資組合中提供較好的 diversification benefit。

---

## 三、投資組合最適化

本次透過蒙地卡羅模擬法建立投資組合機會集合，
並從所有模擬組合中找出年化波動率最低的投資組合，
將其定義為 Minimum Variance Portfolio（MVP）。

### MVP 資產配置建議

| 資產 | MVP 配置權重 |
|---|---:|
{allocation_table}
| 合計 | **{mvp_weights.sum():.2%}** |

### MVP 整體表現

- 預期年化報酬率：**{mvp_return:.2%}**
- 預期年化波動率：**{mvp_volatility:.2%}**

### Portfolio Opportunity Set
"""))

# ---------- 四、Opportunity Set：放在 MVP 分析段落內 ----------

fig_opportunity, ax_opportunity = plt.subplots(figsize=(10, 6))

scatter = ax_opportunity.scatter(
    portfolio_results["Annualized Volatility"],
    portfolio_results["Expected Annual Return"],
    c=portfolio_results["Expected Annual Return"],
    cmap="viridis",
    alpha=0.55,
    s=15
)

# 標示 MVP
ax_opportunity.scatter(
    mvp_volatility,
    mvp_return,
    color="red",
    marker="*",
    s=350,
    edgecolors="black",
    linewidths=1.2,
    label="Minimum Variance Portfolio (MVP)",
    zorder=5
)

ax_opportunity.annotate(
    f"MVP\nReturn: {mvp_return:.2%}\nVolatility: {mvp_volatility:.2%}",
    xy=(mvp_volatility, mvp_return),
    xytext=(15, 15),
    textcoords="offset points",
    color="red",
    fontsize=10,
    bbox=dict(
        boxstyle="round,pad=0.4",
        facecolor="white",
        edgecolor="red",
        alpha=0.85
    )
)

ax_opportunity.set_title("Portfolio Opportunity Set")
ax_opportunity.set_xlabel("Annualized Volatility")
ax_opportunity.set_ylabel("Expected Annual Return")
ax_opportunity.legend()
ax_opportunity.grid(alpha=0.3)

fig_opportunity.colorbar(
    scatter,
    ax=ax_opportunity,
    label="Expected Annual Return"
)

fig_opportunity.tight_layout()
plt.show()

print("✅ Portfolio Opportunity Set 已成功產生。")
print("✅ MVP 已標示於散佈圖中。")


# ---------- 五、結論 ----------

display(Markdown(f"""
### 結論

由投資組合機會集合可以觀察到，不同資產配置會產生不同的報酬率與風險組合。

本次模擬結果中的 MVP 配置，其預期年化報酬率為 **{mvp_return:.2%}**，
預期年化波動率為 **{mvp_volatility:.2%}**。

MVP 透過配置不同風險特性的資產，
利用資產間的相關性差異來降低投資組合的整體波動率。

本次 MVP 為蒙地卡羅模擬結果中的近似解，
不代表未來一定能達到相同的報酬率或波動率。
實際投資時仍應定期更新資料並重新評估資產配置。
"""))